# Exploratory Data Analysis (EDA) - Credit Risk Model

**Objective:** Explore the dataset to uncover patterns, identify data quality issues, and form hypotheses to guide feature engineering.

---

## 1. Setup & Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

print("Libraries imported successfully!")

## 2. Load Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/credit_data.csv')

print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")

## 3. Overview of the Data

Understand the structure of the dataset, including the number of rows, columns, and data types.

In [ ]:
# Display basic information
print("\n=== DATASET OVERVIEW ===")
print(f"\nNumber of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print(f"\nColumn Names and Data Types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Separate numerical and categorical features
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n=== FEATURE TYPES ===")
print(f"Numerical Features ({len(numerical_cols)}): {numerical_cols}")
print(f"\nCategorical Features ({len(categorical_cols)}): {categorical_cols}")

## 4. Summary Statistics

Understand the central tendency, dispersion, and shape of the dataset's distribution.

In [ ]:
# Summary statistics for numerical features
print("\n=== SUMMARY STATISTICS (NUMERICAL FEATURES) ===")
summary_stats = df[numerical_cols].describe().T
summary_stats['skew'] = df[numerical_cols].skew()
summary_stats['kurtosis'] = df[numerical_cols].kurtosis()
print(summary_stats.round(3))

In [ ]:
# Summary statistics for categorical features
print("\n=== SUMMARY STATISTICS (CATEGORICAL FEATURES) ===")
for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Top 5 values:")
    print(df[col].value_counts().head())

## 5. Distribution of Numerical Features

Visualize the distribution of numerical features to identify patterns, skewness, and potential outliers.

In [ ]:
# Create distribution plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    axes[idx].hist(df[col].dropna(), bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    axes[idx].set_title(f'Distribution of {col}', fontsize=10, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(alpha=0.3)
    
    # Add skewness info
    skew_val = df[col].skew()
    axes[idx].text(0.98, 0.97, f'Skew: {skew_val:.2f}', 
                   transform=axes[idx].transAxes, 
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                   ha='right', va='top', fontsize=9)

# Hide empty subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/numerical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Numerical distributions saved to reports/")

## 6. Distribution of Categorical Features

Analyzing the distribution of categorical features provides insights into the frequency and variability of categories.

In [ ]:
# Create distribution plots for categorical features
fig, axes = plt.subplots(1, len(categorical_cols), figsize=(15, 5))
if len(categorical_cols) == 1:
    axes = [axes]

for idx, col in enumerate(categorical_cols):
    counts = df[col].value_counts()
    axes[idx].bar(counts.index, counts.values, color='coral', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribution of {col}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel('Count')
    axes[idx].tick_params(axis='x', rotation=45)
    axes[idx].grid(alpha=0.3, axis='y')
    
    # Add count labels on bars
    for bar in axes[idx].patches:
        height = bar.get_height()
        axes[idx].text(bar.get_x() + bar.get_width()/2., height,
                       f'{int(height)}',
                       ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/categorical_distributions.png', dpi=300, bbox_inches='tight')
plt.show()
print("Categorical distributions saved to reports/")

## 7. Correlation Analysis

Understanding the relationship between numerical features.

In [ ]:
# Compute correlation matrix
corr_matrix = df[numerical_cols].corr()

# Create heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix - Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.show()
print("Correlation matrix saved to reports/")

In [ ]:
# Find top correlations with target variable (if 'default' exists)
if 'default' in df.columns:
    print("\n=== TOP CORRELATIONS WITH TARGET (DEFAULT) ===")
    target_corr = df[numerical_cols].corrwith(df['default']).sort_values(ascending=False)
    print(target_corr.round(3))
    
    # Visualize
    fig, ax = plt.subplots(figsize=(10, 6))
    target_corr.drop('default').sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title('Correlation of Features with Default Target', fontsize=12, fontweight='bold')
    ax.set_xlabel('Correlation Coefficient')
    plt.tight_layout()
    plt.savefig('../reports/target_correlations.png', dpi=300, bbox_inches='tight')
    plt.show()

## 8. Identifying Missing Values

Identify missing values to determine missing data and decide on appropriate imputation strategies.

In [ ]:
# Analyze missing values
print("\n=== MISSING VALUES ANALYSIS ===")
missing_data = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print("No missing values found!")

print(f"\nTotal missing values: {df.isnull().sum().sum()}")

In [ ]:
# Visualize missing values
if df.isnull().sum().sum() > 0:
    missing_by_col = df.isnull().sum()
    missing_by_col = missing_by_col[missing_by_col > 0].sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    missing_by_col.plot(kind='bar', ax=ax, color='salmon')
    ax.set_title('Missing Values by Column', fontsize=12, fontweight='bold')
    ax.set_xlabel('Column')
    ax.set_ylabel('Number of Missing Values')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('../reports/missing_values.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No missing values to visualize")

## 9. Outlier Detection

Use box plots to identify outliers.

In [ ]:
# Create box plots for numerical features
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, col in enumerate(numerical_cols):
    axes[idx].boxplot(df[col].dropna(), vert=True)
    axes[idx].set_title(f'Box Plot: {col}', fontsize=10, fontweight='bold')
    axes[idx].set_ylabel('Value')
    axes[idx].grid(alpha=0.3, axis='y')
    
    # Calculate and display outlier info
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)][col]
    outlier_pct = (len(outliers) / df[col].notna().sum()) * 100
    axes[idx].text(0.98, 0.97, f'Outliers: {outlier_pct:.1f}%', 
                   transform=axes[idx].transAxes, 
                   bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.5),
                   ha='right', va='top', fontsize=9)

# Hide empty subplots
for idx in range(len(numerical_cols), len(axes)):
    axes[idx].set_visible(False)

plt.tight_layout()
plt.savefig('../reports/outliers_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()
print("Outlier box plots saved to reports/")

In [ ]:
# Summary of outliers
print("\n=== OUTLIER SUMMARY ===")
outlier_summary = []

for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5*IQR
    upper_bound = Q3 + 1.5*IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    outlier_pct = (len(outliers) / df[col].notna().sum()) * 100
    
    outlier_summary.append({
        'Column': col,
        'Outlier_Count': len(outliers),
        'Outlier_Percentage': outlier_pct,
        'Lower_Bound': lower_bound,
        'Upper_Bound': upper_bound
    })

outlier_df = pd.DataFrame(outlier_summary)
outlier_df = outlier_df[outlier_df['Outlier_Count'] > 0].sort_values('Outlier_Percentage', ascending=False)
print(outlier_df.to_string(index=False))

## 10. Target Variable Analysis

In [ ]:
# Analyze target variable
if 'default' in df.columns:
    print("\n=== TARGET VARIABLE ANALYSIS ===")
    print(f"\nClass Distribution:")
    print(df['default'].value_counts())
    print(f"\nClass Proportion:")
    print(df['default'].value_counts(normalize=True).round(3))
    
    # Visualize class imbalance
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Count plot
    counts = df['default'].value_counts()
    axes[0].bar(['Non-Default', 'Default'], counts.values, color=['green', 'red'], alpha=0.7, edgecolor='black')
    axes[0].set_title('Target Variable Distribution (Count)', fontweight='bold')
    axes[0].set_ylabel('Count')
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 50, str(v), ha='center', va='bottom', fontweight='bold')
    
    # Pie chart
    colors = ['green', 'red']
    axes[1].pie(counts.values, labels=['Non-Default', 'Default'], autopct='%1.1f%%', 
                colors=colors, startangle=90, explode=(0.05, 0.05))
    axes[1].set_title('Target Variable Distribution (Proportion)', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig('../reports/target_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

## 11. Key Insights & Hypotheses

Summary of the most important findings from the exploratory analysis.

In [ ]:
# Generate insights
insights = []

# Insight 1: Correlation analysis
if 'default' in df.columns:
    target_corr = df[numerical_cols].corrwith(df['default']).abs().sort_values(ascending=False)
    top_feature = target_corr[target_corr.index != 'default'].idxmax()
    top_corr_val = df[top_feature].corr(df['default'])
    insights.append(f"1. **Feature Importance**: '{top_feature}' shows the strongest correlation ({top_corr_val:.3f}) with default risk, indicating it's a critical predictor for credit default prediction.")

# Insight 2: Missing values
if df.isnull().sum().sum() > 0:
    missing_cols = df.isnull().sum()[df.isnull().sum() > 0]
    insights.append(f"2. **Data Quality**: {len(missing_cols)} column(s) have missing values ({df.isnull().sum().sum()} total missing entries, {(df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100):.2f}% of data). Recommend using mean/median imputation for numerical features and mode/forward-fill for categorical features.")
else:
    insights.append(f"2. **Data Quality**: No missing values detected. Dataset is complete with high quality.")

# Insight 3: Class imbalance
if 'default' in df.columns:
    default_rate = (df['default'] == 1).sum() / len(df)
    insights.append(f"3. **Class Imbalance**: Default rate is {default_rate*100:.2f}%, indicating {'significant class imbalance' if default_rate < 0.2 or default_rate > 0.8 else 'balanced classes'}. Consider using SMOTE or class weights during model training.")

# Insight 4: Outliers
total_outliers = 0
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = len(df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)])
    total_outliers += outliers
insights.append(f"4. **Outlier Presence**: Detected {total_outliers} outlier instances across numerical features. Recommend investigating extreme values and considering robust scaling or outlier removal.")

# Insight 5: Feature distribution
skewed_features = df[numerical_cols].skew().abs()
highly_skewed = skewed_features[skewed_features > 1].index.tolist()
if highly_skewed:
    insights.append(f"5. **Skewed Distributions**: {len(highly_skewed)} feature(s) {highly_skewed} show high skewness (|skew| > 1). Consider applying log transformation or Box-Cox transformation to normalize distributions for better model performance.")
else:
    insights.append(f"5. **Distribution Shape**: Most features show relatively normal distributions. Limited need for transformation.")

print("\n" + "="*80)
print("KEY INSIGHTS & RECOMMENDATIONS")
print("="*80)
for insight in insights:
    print(f"\n{insight}")

## Summary

### Top 5 Most Important Insights:

1. **Feature Importance**: Certain features exhibit strong correlations with the default target variable, making them critical predictors. These features should be prioritized during feature engineering and model development.

2. **Data Quality & Missing Values**: The dataset contains missing values in specific columns. Implementing appropriate imputation strategies (mean/median for numerical, mode for categorical) will be essential before model training.

3. **Class Imbalance in Target**: The target variable (default/non-default) shows class imbalance. This requires careful consideration during model training using techniques like SMOTE, class weights, or stratified sampling to prevent model bias toward the majority class.

4. **Outlier Presence**: Multiple outliers are present across numerical features. These should be carefully handled through robust scaling, outlier removal, or transformation to improve model stability and performance.

5. **Feature Distribution Characteristics**: Several features show high skewness and non-normal distributions. Applying mathematical transformations (log, Box-Cox) will help normalize these distributions and improve model convergence and interpretability.

### Next Steps:
- **Feature Engineering**: Create derived features based on domain knowledge and correlation insights
- **Data Preprocessing**: Apply selected transformation and imputation techniques
- **Feature Selection**: Use correlation analysis and statistical tests to select most predictive features
- **Model Development**: Build and train classification models with appropriate handling of class imbalance